<a href="https://colab.research.google.com/github/divyasri2609/Medical-Transcriptions/blob/main/RAG_%2B_AGENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import re
import os
import zipfile
!pip install faiss-cpu > /dev/null
import faiss

!pip install sentence-transformers > /dev/null
from sentence_transformers import SentenceTransformer

from google.colab import files
uploaded = files.upload()

zip_file = list(uploaded.keys())[0]
print("Uploaded file:", zip_file)


extract_path = "data"

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Files extracted!")

csv_path = None

for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.endswith(".csv"):
            csv_path = os.path.join(root, file)

print("CSV found at:", csv_path)

df = pd.read_csv(csv_path, encoding='latin-1')

df = df[['transcription', 'medical_specialty']]
df = df.dropna()

print("Dataset shape:", df.shape)

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

df['clean_text'] = df['transcription'].apply(clean_text)

top_classes = df['medical_specialty'].value_counts().head(10).index
df = df[df['medical_specialty'].isin(top_classes)]

df = df[df['clean_text'].str.len() > 50]

print("Cleaned dataset:", df.shape)

print("\nLoading embedding model...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

texts = df['clean_text'].tolist()

print("Generating embeddings...")
embeddings = embed_model.encode(texts, show_progress_bar=True)

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("Total vectors indexed:", index.ntotal)

def retrieve(query, k=5):
    query_clean = clean_text(query)
    query_vec = embed_model.encode([query_clean])

    distances, indices = index.search(query_vec, k)

    return df.iloc[indices[0]], distances[0]

def predict_specialty(retrieved_df):
    return retrieved_df['medical_specialty'].value_counts().idxmax()

def generate_insight(retrieved_df):
    return "Retrieved cases indicate patterns consistent with the predicted specialty."

def compare_cases(query, retrieved_df):
    return "The query shares clinical similarities with retrieved cases but may differ in severity or duration."

def clinical_agent(query):

    reasoning_trace = []

    reasoning_trace.append("Step 1: Analyzing query")

    # Removed the query length check to allow shorter queries to be processed
    # if len(query.split()) < 5:
    #     reasoning_trace.append("Query too short")
    #     return {
    #         "response": "⚠️ Please provide more detailed clinical information.",
    #         "trace": reasoning_trace
    #     }

    reasoning_trace.append("Step 2: Retrieving similar cases")
    retrieved_df, distances = retrieve(query)

    if retrieved_df.empty:
        reasoning_trace.append("No results found")
        return {
            "response": "⚠️ No similar cases found.",
            "trace": reasoning_trace
        }

    reasoning_trace.append("Step 3: Predicting specialty")
    specialty = predict_specialty(retrieved_df)

    reasoning_trace.append("Step 4: Generating insights")
    insight = generate_insight(retrieved_df)

    reasoning_trace.append("Step 5: Comparing cases")
    comparison = compare_cases(query, retrieved_df)

    confidence = round(1 / (1 + np.mean(distances)), 2)
    reasoning_trace.append(f"Step 6: Confidence = {confidence}")

    response = f"""
🔹 Predicted Specialty: {specialty}

🔹 Confidence Score: {confidence}

🔹 Clinical Insight:
{insight}

🔹 Comparison:
{comparison}

🔹 Evidence:
Top {len(retrieved_df)} similar cases retrieved.
"""

    return {
        "response": response,
        "trace": reasoning_trace
    }

query = "patient has chest pain and shortness of breath for two days"

result = clinical_agent(query)

print("\n===== FINAL RESPONSE =====")
print(result['response'])

print("\n===== REASONING TRACE =====")
for step in result['trace']:
    print(step)

queries = [
    "brain tumor symptoms with headache",
    "bone fracture after accident",
    "skin rash infection",
    "kidney failure symptoms",
    "heart pain and breathing problem"
]

for q in queries:
    print("\n====================")
    print("Query:", q)
    print(clinical_agent(q)['response'])

print("\n=== EDGE CASE TEST ===")
print(clinical_agent("fever"))

Saving archive (10).zip to archive (10) (1).zip
Uploaded file: archive (10) (1).zip
Files extracted!
CSV found at: data/mtsamples.csv
Dataset shape: (4966, 2)
Cleaned dataset: (3613, 3)

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings...


Batches:   0%|          | 0/113 [00:00<?, ?it/s]

Total vectors indexed: 3613

===== FINAL RESPONSE =====

🔹 Predicted Specialty:  Consult - History and Phy.

🔹 Confidence Score: 0.5899999737739563

🔹 Clinical Insight:
Retrieved cases indicate patterns consistent with the predicted specialty.

🔹 Comparison:
The query shares clinical similarities with retrieved cases but may differ in severity or duration.

🔹 Evidence:
Top 5 similar cases retrieved.


===== REASONING TRACE =====
Step 1: Analyzing query
Step 2: Retrieving similar cases
Step 3: Predicting specialty
Step 4: Generating insights
Step 5: Comparing cases
Step 6: Confidence = 0.5899999737739563

Query: brain tumor symptoms with headache

🔹 Predicted Specialty:  Neurology

🔹 Confidence Score: 0.5400000214576721

🔹 Clinical Insight:
Retrieved cases indicate patterns consistent with the predicted specialty.

🔹 Comparison:
The query shares clinical similarities with retrieved cases but may differ in severity or duration.

🔹 Evidence:
Top 5 similar cases retrieved.


Query: bone fr